In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# ACC cue-on band power (overlap window)

Load ACC `cue_on` TFRs, compute time-averaged power in each frequency band over **(0.37, 0.60) s** per trial and electrode, and merge with trialwise variables from `combined_df`.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from membandit.behavior import load_behavior_tables, preprocess_combined_df

warnings.filterwarnings('ignore')

# ---- Config ----
base_dir = Path('/Volumes/T7 Shield')
load_dir = base_dir / 'work/qasims01/MemoryBanditData/EMU'
ephys_root = base_dir / 'projects/guLab/Salman/EphysAnalyses'

EVENT = 'cue_on'
CONDITION = 'Day1'
ROI = 'ACC'
TIME_WINDOW = (0.37, 0.60)
# (0.37, 0.60)
# (0.60, 1)

FBANDS = ['high_theta', 'hfa']

In [ ]:
_, combined_df_raw = load_behavior_tables(str(load_dir))
combined_df = preprocess_combined_df(combined_df_raw)
combined_df = combined_df[combined_df['condition'] == CONDITION].copy()

print('combined_df shape:', combined_df.shape)
print('participants:', combined_df['participant'].nunique())

In [ ]:
def discover_acc_subjects() -> list[str]:
    """Day1 subjects with cue_on TFR and at least one ACC electrode."""
    subjects = []
    if not ephys_root.exists():
        return subjects
    for subj_dir in sorted(ephys_root.iterdir()):
        if not subj_dir.is_dir():
            continue
        subject = subj_dir.name
        tfr_path = subj_dir / 'scratch' / 'TFR' / f'{EVENT}-tfr.h5'
        elec_path = subj_dir / 'Day1_reref_elec_df'
        if not tfr_path.exists() or not elec_path.exists():
            continue
        elec_df = pd.read_csv(elec_path)
        if 'salman_region' not in elec_df.columns:
            continue
        if (elec_df['salman_region'] == ROI).any():
            subjects.append(subject)
    return subjects


def get_acc_channels(subject: str) -> list[str]:
    elec_path = ephys_root / subject / 'Day1_reref_elec_df'
    elec_df = pd.read_csv(elec_path)
    return elec_df.loc[elec_df['salman_region'] == ROI, 'label'].astype(str).tolist()


def assign_fband(freq: float) -> str:
    if freq < 4:
        return 'low_theta'
    if freq < 8:
        return 'high_theta'
    if freq < 13:
        return 'alpha'
    if freq < 30:
        return 'beta'
    if freq < 70:
        return 'gamma'
    return 'hfa'


def freq_mask_for_band(freqs: np.ndarray, band: str) -> np.ndarray:
    if band == 'low_theta':
        return freqs < 4
    if band == 'high_theta':
        return (freqs >= 4) & (freqs < 8)
    if band == 'alpha':
        return (freqs >= 8) & (freqs < 13)
    if band == 'beta':
        return (freqs >= 13) & (freqs < 30)
    if band == 'gamma':
        return (freqs >= 30) & (freqs < 70)
    if band == 'hfa':
        return (freqs >= 30) & (freqs < 201)
    raise ValueError(f'Unknown band: {band}')


def build_memory_metadata(subject: str) -> pd.DataFrame:
    memory_df = (
        combined_df[
            (combined_df['participant'] == subject) &
            (combined_df['condition'] == CONDITION)
        ]
        .dropna(subset=['trials_mem'])
        .sort_values('trials_mem')
        .reset_index(drop=True)
    )
    return memory_df


def extract_subject_band_power(subject: str) -> pd.DataFrame | None:
    tfr_path = ephys_root / subject / 'scratch' / 'TFR' / f'{EVENT}-tfr.h5'
    acc_channels = get_acc_channels(subject)
    if len(acc_channels) == 0:
        return None

    tfr_obj = mne.time_frequency.read_tfrs(str(tfr_path))
    tfr = tfr_obj[0] if isinstance(tfr_obj, list) else tfr_obj

    present_channels = [ch for ch in acc_channels if ch in tfr.ch_names]
    if len(present_channels) == 0:
        return None

    tfr = tfr.copy().pick(present_channels)
    n_epochs = len(tfr)
    memory_df = build_memory_metadata(subject)
    if len(memory_df) == 0:
        return None
    if len(memory_df) != n_epochs:
        print(
            f'WARNING {subject}: memory trials ({len(memory_df)}) != TFR epochs ({n_epochs}); skipping'
        )
        return None

    tmin, tmax = TIME_WINDOW
    time_mask = (tfr.times > tmin) & (tfr.times <= tmax)
    if not np.any(time_mask):
        print(f'WARNING {subject}: no TFR samples in window {TIME_WINDOW}; skipping')
        return None

    data = np.asarray(tfr.data, dtype=np.float64)
    data_win = data[:, :, :, time_mask]
    freqs = np.asarray(tfr.freqs, dtype=np.float64)
    trials_mem = memory_df['trials_mem'].to_numpy()

    parts = []
    for band in FBANDS:
        fmask = freq_mask_for_band(freqs, band)
        if not np.any(fmask):
            continue
        band_power = np.nanmean(data_win[:, :, fmask, :], axis=(2, 3))
        part = pd.DataFrame(band_power, columns=tfr.ch_names)
        part['epoch'] = np.arange(n_epochs)
        part = part.melt(id_vars='epoch', var_name='channel', value_name='power')
        part['fband'] = band
        parts.append(part)

    out = pd.concat(parts, ignore_index=True)
    out['trials_mem'] = trials_mem[out['epoch'].to_numpy(dtype=int)]
    out['participant'] = subject
    return out.drop(columns=['epoch'])

MNI_COLUMNS = ('mni_x', 'mni_y', 'mni_z')

def build_acc_elec_lookup(subjects: list[str]) -> pd.DataFrame:
    """One row per ACC electrode with MNI coordinates."""
    parts = []
    for subject in subjects:
        elec_path = ephys_root / subject / 'Day1_reref_elec_df'
        elec_df = pd.read_csv(elec_path)
        acc = elec_df.loc[elec_df['salman_region'] == ROI].copy()
        missing = [c for c in MNI_COLUMNS if c not in acc.columns]
        if missing:
            print(f'WARNING {subject}: elec_df missing {missing}; skipping MNI for this subject')
            continue
        acc['participant'] = subject
        keep = ['participant', 'label', *MNI_COLUMNS]
        if 'hemi' in acc.columns:
            keep.append('hemi')
        parts.append(acc[keep])
    if not parts:
        raise RuntimeError('No ACC electrode MNI rows found; check Day1_reref_elec_df columns.')
    lookup = pd.concat(parts, ignore_index=True)
    lookup = lookup.rename(columns={'label': 'channel'})
    return lookup.drop_duplicates(subset=['participant', 'channel'])


acc_subjects = discover_acc_subjects()
print('ACC subjects with cue_on TFR:', len(acc_subjects))
print(acc_subjects)

In [ ]:
power_rows = []
skipped = []
for subject in tqdm(acc_subjects, desc='Extracting ACC band power'):
    subj_df = extract_subject_band_power(subject)
    if subj_df is None or len(subj_df) == 0:
        skipped.append(subject)
        continue
    power_rows.append(subj_df)

if len(power_rows) == 0:
    raise RuntimeError('No band-power rows extracted; check paths and subject list.')

band_power_df = pd.concat(power_rows, ignore_index=True)

acc_elec_lookup = build_acc_elec_lookup(acc_subjects)
band_power_df = band_power_df.merge(
    acc_elec_lookup,
    on=['participant', 'channel'],
    how='left',
)

acc_cue_band_df = band_power_df.merge(
    combined_df,
    on=['participant', 'trials_mem'],
    how='left',
)

dup_cols = [c for c in acc_cue_band_df.columns if c.endswith('_dup')]
if dup_cols:
    acc_cue_band_df = acc_cue_band_df.drop(columns=dup_cols)

print('acc_elec_lookup shape:', acc_elec_lookup.shape)
print('Skipped subjects:', skipped)
print('band_power_df shape:', band_power_df.shape)
print('acc_cue_band_df shape:', acc_cue_band_df.shape)

In [ ]:
n_subjects = acc_cue_band_df['participant'].nunique()
n_channels = acc_cue_band_df['channel'].nunique()
n_trials = acc_cue_band_df.groupby(['participant', 'trials_mem']).ngroups

print('Subjects:', n_subjects)
print('Unique channels (across subjects):', n_channels)
print('Participant x trial groups:', n_trials)
print('Rows per fband:')
print(acc_cue_band_df['fband'].value_counts().sort_index())

missing_behavior = acc_cue_band_df['memory_outcome'].isna().sum() if 'memory_outcome' in acc_cue_band_df.columns else np.nan
print('Rows missing memory_outcome after merge:', missing_behavior)

display(acc_cue_band_df.head(12))
display(acc_cue_band_df[['participant', 'trials_mem', 'channel', 'fband', 'power', 'memory_outcome', 'zrpe', 'rpe_cat']].head(12))

In [ ]:
acc_cue_band_df['unique_channel'] = acc_cue_band_df['participant'] + '_' + acc_cue_band_df['channel']

In [ ]:
acc_cue_band_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare grouped means for barplot (same as before)
bar_df = acc_cue_band_df[acc_cue_band_df.fband=='hfa'].groupby(
    ['unique_channel', 'fband', 'memory_outcome', 'rpe_cat', 'hits']
).mean(numeric_only=True).reset_index()

# For the stripplot overlay, use the same groupby as the barplot ("one dot per group")
strip_df = bar_df

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=bar_df,
    x='hits', y='power', hue='rpe_cat', errorbar='se',
    alpha=0.8, dodge=True
)

# Overlay stripplot, with horizontal jitter
sns.stripplot(
    data=strip_df,
    x='hits', y='power', hue='rpe_cat',
    dodge=True, jitter=0.24, alpha=0.7, ax=ax,
    marker='o', edgecolor='gray', linewidth=0.5, zorder=10,
    legend=False
)

plt.legend(title='rpe_cat')
plt.tight_layout()
plt.show()




In [ ]:
import statsmodels.formula.api as smf
model = smf.mixedlm(formula="power ~ 1 + memory_outcome * C(rpe_cat, Treatment(reference='pos')) + mni_x + mni_y + mni_z", 
groups='unique_channel', 
data=acc_cue_band_df[acc_cue_band_df.fband=='hfa'].dropna(subset=['power', 'memory_outcome', 'rpe_cat'])).fit()
print(model.summary())

In [ ]:
model = smf.mixedlm(formula="power ~ 1 + C(rpe_cat, Treatment(reference='pos')) * mni_z", 
groups='unique_channel', 
data=acc_cue_band_df[acc_cue_band_df.fband=='hfa'].dropna(subset=['power', 'memory_outcome', 'rpe_cat'])).fit()
print(model.summary())

In [ ]:
all_pos_neg_intx = []
all_pos_neu_intx = []
channel_df = acc_cue_band_df.groupby('unique_channel').mean(numeric_only=True).reset_index()

for channel in acc_cue_band_df['unique_channel'].unique():
    subj_df = acc_cue_band_df[acc_cue_band_df['unique_channel'] == channel]
    model = smf.ols(formula="power ~ 1 + memory_outcome * C(rpe_cat, Treatment(reference='pos')) +", 
    data=subj_df[subj_df.fband=='hfa'].dropna(subset=['power', 'memory_outcome', 'rpe_cat'])).fit()
    pos_neg_intx = model.params.values[-2]
    pos_neu_intx = model.params.values[-1]
    all_pos_neg_intx.append(pos_neg_intx)
    all_pos_neu_intx.append(pos_neu_intx)

channel_df['pos_neg_intx'] = all_pos_neg_intx
channel_df['pos_neu_intx'] = all_pos_neu_intx


In [ ]:
sns.regplot(data=channel_df, x='mni_x', y='pos_neg_intx')
sns.regplot(data=channel_df, x='mni_x', y='pos_neu_intx')

In [ ]:
sns.displot(all_pos_neg_intx)
sns.displot(all_pos_neu_intx)